# Identify Potential Neighborhood Nodes

This notebook provides a way to see Trello cards for a particular neighborhood for people who have expressed interest, but haven't been installed or surveyed yet. This is helpful for identifying other people to contact when organizing a Meshraising event.

In [1]:
import json
from pathlib import Path

from IPython.display import display
import ipywidgets as widgets
from lonboard import viz
import pandas as pd
import geopandas as gpd
from pyogrio.errors import DataSourceError
import requests

In [2]:
DATA_DIR = Path.cwd() / "data" 

In [3]:
DATA_DIR_SRC = DATA_DIR / "source"

In [4]:
def fetch_neighborhoods_geojson() -> dict:
    url = "https://gis.tucsonaz.gov/public/rest/services/PublicMaps/NeighborhoodsPlans/MapServer/11/query"
    params = {
        "where": "1=1",
        "f": "geojson",
        "spatialRefId": 4326,
    }
    resp = requests.get(url, params=params)
    return resp.json()

In [5]:
def load_neighborhoods(path: Path) -> gpd.GeoDataFrame:
    try:
        neighborhoods = gpd.read_file(path)
    
    except DataSourceError:
        neighborhoods_json = fetch_neighborhoods_geojson()
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w") as f:
            json.dump(neighborhoods_json, f)
    
        neighborhoods = gpd.read_file(path)

    return neighborhoods.rename(columns=lambda x: x.lower())

In [6]:
neighborhoods = load_neighborhoods(DATA_DIR_SRC / "neighborhoods.json")

In [7]:
# Load nodes exported from Trello with trello-to-geojson
nodes = gpd.read_file(DATA_DIR_SRC / "nodes.geojson")

In [8]:
# Filter nodes to ones that haven't been surveyed yet
nodes_pending = nodes[pd.col("status").isin(["Purgatory", "Submitted Requests", "Ready for Contact & Survey"])]

In [9]:
# Display a lookup for the neighborhood names and then display the potential nodes on a simple map as well as a table once a neighborhood has been selected
neighborhood_select = widgets.Combobox(
    placeholder="Choose Neighborhood",
    options=neighborhoods["name"].to_list(),
    ensure_option=True,
    disabled=False
)
out = widgets.Output()

def handle_select_change(change):
    out.clear_output()
    neighborhood = neighborhoods[pd.col("name") == change.new]
    neighborhood_nodes = nodes_pending.sjoin(
        neighborhood,
        how="inner",
        predicate="intersects"
    )
    with out:
        display(viz(neighborhood_nodes))
        display(neighborhood_nodes)
    

neighborhood_select.observe(handle_select_change, names='value')

display(neighborhood_select, out)

Combobox(value='', ensure_option=True, options=('Coyote Corridor East', 'Silver Hills Estates', 'Harlan Height…

Output()